# driver_DD_NMROM
Driver to implement and test NM ROM on the 2D Burgers Equation.  

In [ ]:
import sys
with open("./../../../PATHS.txt") as file:
  paths = file.read().splitlines()
sys.path.extend(paths)

In [ ]:
from dd_nm_rom import env
env.set(
  backend="numpy",
  device="cpu",
  device_idx=0,
  nb_threads=4,
  epsilon=1e-10,
  floatx="float64",
  seed=0
)

In [ ]:
import os
import json
import numpy as np
import dill as pickle
import matplotlib.pyplot as plt
import dill as pickle

from matplotlib import cm
import numpy_indexed as npi

In [ ]:
from dd_nm_rom import ops
from dd_nm_rom import utils
from dd_nm_rom import postproc
from dd_nm_rom import fom as fom_mod
from dd_nm_rom import rom as rom_mod
from dd_nm_rom import field as field_mod
from dd_nm_rom.elements import mesh as mesh_mod

In [ ]:
mesh_inp = {
  "nx_intr": 48,
  "ny_intr": 48,
  "lx_sub": 0.5,
  "ly_sub": 0.5,
  "x0": 0.0,
  "y0": 0.0,
  "n_sub_x": 2,
  "n_sub_y": 2,
  "with_bounds": True
}
nu = 0.001

In [ ]:
paths = {
  "pod_dir": "/g/g92/zanardi1/Workspace/Codes/DD-NM-ROM/run/unsteady/dset.2by2/pod/run.02/snapshots/",
  "nets_dir": "/g/g92/zanardi1/Workspace/Codes/DD-NM-ROM/run/unsteady/dset.2by2/nets/run.01.n2000/",
  "nets_tag": {
    "interior": "interior_ld_24_rnz_10",
    "port": "port_ld_10_rnz_8"
  },
  "figs_dir": "/g/g92/zanardi1/Workspace/Codes/DD-NM-ROM/run/unsteady/dset.2by2/figures/test_hr/",
  "test_case": "/g/g92/zanardi1/Workspace/Codes/DD-NM-ROM/run/unsteady/dset.2by2/datagen/n2000/test/case_00006.p"
}
for k in ("figs_dir",):
  os.makedirs(paths[k], exist_ok=True)

## Solve DD FOM

In [ ]:
# Mesh
mesh = mesh_mod.MeshDD(**mesh_inp)
mesh.build()
X, Y = mesh.grid
xx, yy = X.flatten(), Y.flatten()
# Field
field = field_mod.SinMultiPeak(
  mesh=mesh,
  mu_lim=[0.5, 1.5],
  forced_config=[1,0,0,0],
  bc_type="periodic"
)
field.set_params(mu=field.sample_design_space())
# FOM
fom = fom_mod.Burgers2D(mesh=mesh, nu=nu)
fom.build(field)
# DD-FOM
dd_fom = fom_mod.DDBurgers2D(monolithic=fom, constraint_type='strong', scaling=-1)
dd_fom.build()

In [ ]:
path_to_nets = {}
for (element, tag) in paths["nets_tag"].items():
  suffix = f"/{tag}/merged/{element}/"
  path_to_nets[element] = paths["nets_dir"] + suffix
nn_configfiles = rom_mod.nonlinear.domain_dec.load_nn_configfiles(
  mesh=mesh, dd_fom=dd_fom, path_to_nets=path_to_nets
)

Test case set up

In [ ]:
test_case = pickle.load(open(paths["test_case"], "rb"))

In [ ]:
x0 = test_case["snapshots"][0]
solver = test_case["solver"]
uv_fom = test_case["solution"]
runtime_fom = test_case["runtime"]
# Time instants plotted
teval = test_case["time"].squeeze()[::10]
ieval = np.arange(len(test_case["time"]))[::10]

DD-FOM
> Building

In [ ]:
field.set_params(test_case["mu"])
fom.build(field)
dd_fom.build()

> Solution

In [ ]:
path_ij = paths["figs_dir"] + "/fom/"
# Statistics
os.makedirs(path_ij, exist_ok=True)
with open(path_ij + "/runtime.json", "w") as file:
  json.dump(runtime_fom, file, indent=2)
# # Postprocessing
# for i in ieval:
#   postproc.plot_field_fom_rom(
#     path=path_ij,
#     mesh=mesh,
#     uv_fom=uv_fom,
#     uv_rom=None,
#     index=i,
#     show_labels=False
#   )
# postproc.animate_fom_rom(path_ij, mesh, uv_fom)

DD-NM-ROM

In [ ]:
hr_active = True
energy_min = 1e-3

In [ ]:
filename = paths["pod_dir"] + "/merged/svd.p"
svd = pickle.load(open(filename, "rb"))["res"][0]

In [ ]:
energy = np.cumsum(svd["s"]**2) / np.sum(svd["s"]**2)
nb_bases = np.where(energy > 1-energy_min)[0].min()+1
res_bases = svd["u"][:,:nb_bases]
hr_n_samples = 2*nb_bases
hr_n_samples

In [ ]:
# Configuration
rom_cfg = "srpc"
if hr_active:
  rom_cfg += "_hr"
# Building
dd_rom = rom_mod.DD_NM_ROM(
  dd_fom=dd_fom,
  nn_configfiles=nn_configfiles,
  res_bases=res_bases,
  hr_active=hr_active,
  hr_n_samples=hr_n_samples,
  hr_n_edge_samples_ratio=0.5,
  hr_sample_small_ports=True,
  hr_small_ports_dim=5,
  constraint_type="strong",
  n_constraints_weak=-1,
  scaling=-1
)

In [ ]:
# Saving path
path_ij = paths["figs_dir"] + f"/rom/{rom_cfg}/"
os.makedirs(path_ij, exist_ok=True)

In [ ]:
# for sub in dd_rom.subdomains:
#   i = sub.hr_nodes_res["u"]
#   j = sub.sub_fom.elem_states["res"].nodes_state[i]
#   plt.scatter(xx[j], yy[j], s=10)
# plt.xlim(mesh.phylim["x"])
# plt.ylim(mesh.phylim["y"])

In [ ]:
postproc.plot_hr_nodes(
  mesh=mesh,
  dd_rom=dd_rom,
  path=path_ij
)

> Solution

In [ ]:
solver["verbose"] = True
uv_rom, *_, iconverged = dd_rom.solve(
  x0=dd_rom.get_init_sol(x=x0),
  runtime=0.0,
  use_guess=False,
  **solver
)

> Statistics

In [ ]:
iruntime = dd_rom.runtime
ierror = dd_rom.compute_error(
  uv_fom, uv_rom, scaling=True, relative=False, axis=0
)

# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
if "linalg" in runtime_fom:
  runtime_fom["lin_solve"] = runtime_fom.pop("linalg")
if "rhs_jac" in runtime_fom:
  runtime_fom["res_jac"] = runtime_fom.pop("rhs_jac")
# <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<

ispeedup = {k: tk/iruntime[k] for (k, tk) in runtime_fom.items()}

> Postprocessing

In [ ]:
for i in ieval:
  postproc.plot_field_fom_rom(
    path=path_ij,
    mesh=mesh,
    uv_fom=uv_fom,
    uv_rom=uv_rom,
    index=i,
    show_labels=False
  )
# postproc.animate_fom_rom(path_ij, mesh, uv_fom, uv_rom)